# Border effects in a plant meristem cell network (Jackson et al. 2019)

**Paper.** Jackson, Duran-Nebreda, Kierzkowski, …, Hamant, Smith, Johnston & Bassel (2019). *Global topological order emerges through local mechanical control of cell divisions in the Arabidopsis shoot apical meristem.* Cell Systems 8:53–65. Data: [osf.io/mxgrh](https://osf.io/mxgrh/).

**What they did.** 3D confocal stacks of the shoot apical meristem (SAM), top four cell layers segmented, cells as nodes and shared walls as edges. Betweenness centrality was computed on the full segmented network (~1,000–1,350 cells), but reported only for ~130–190 *central* cells, to avoid boundary artefacts from the limited field of view. One downstream result: dividing cells have higher betweenness.

**Question here.**
1. How strong is the border effect on betweenness, and does their central-region buffer remove it?
2. Does the betweenness–division association survive a BOSPERRUS correction against the distance to the image edge?
3. Crop experiment: if the field of view were smaller, does BOSPERRUS recover the full-network betweenness better than the raw scores?

Cell positions are not in the published CSVs; they come from the segmentation stacks via `src/physically_realized_graphs/jackson_centroids.py`.

In [ ]:
import zipfile
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.spatial import ConvexHull
from scipy.stats import mannwhitneyu, spearmanr
from shapely.geometry import Point, Polygon

from bosperrus import Flow

ROOT = Path("../..")
DATA_DIR = ROOT / "data" / "jackson2019"
CENTROIDS = ROOT / "results" / "physically_realized_graphs" / "jackson2019" / "centroids.csv"
GENOTYPE, ZIP = "WT", DATA_DIR / "WT_Arabidopsis.zip"
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "svg.fonttype": "none"})

## Load networks, published per-cell values and centroids

`*_network.csv` is the edge list (both directions, with wall areas). `*_topology.csv` holds the published values for the central cells only, and exists for T0 and T1 (division is scored between consecutive time points).

In [ ]:
def read_csv_from_zip(z, name):
    return pd.read_csv(BytesIO(z.read(name)), skipinitialspace=True)


def load_samples(zip_path, genotype):
    centroids = pd.read_csv(CENTROIDS)
    samples = {}
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        for net in sorted(n for n in names if n.endswith("_network.csv")):
            _, rep, time, _ = net.split("/")
            edges = read_csv_from_zip(z, net)
            edges = edges[edges.iloc[:, 2] >= 0.01]  # the paper omits walls < 0.01 µm^2
            G = nx.Graph()
            G.add_edges_from(zip(edges.iloc[:, 0], edges.iloc[:, 1]))
            topo_name = [n for n in names if n.startswith(f"{net.rsplit('/', 1)[0]}/") and n.endswith("_topology.csv")]
            topo = read_csv_from_zip(z, topo_name[0]).set_index("Label") if topo_name else None
            pos = centroids[(centroids.genotype == genotype) & (centroids.rep == rep) & (centroids.time == time)].set_index("label")
            samples[(rep, time)] = dict(G=G, topo=topo, pos=pos.loc[list(G.nodes)])
    return samples


samples = load_samples(ZIP, GENOTYPE)
pd.DataFrame([dict(rep=r, time=t, cells=s["G"].number_of_nodes(), walls=s["G"].number_of_edges(),
                   central=0 if s["topo"] is None else len(s["topo"]),
                   divisions=0 if s["topo"] is None else int(s["topo"].Division.sum()))
              for (r, t), s in samples.items()])

## Distance to the image edge

The SAM is a dome imaged from above. The lateral edge of the segmented region is set by the field of view, so it is an artificial boundary: the tissue continues beyond it. The distance used here is the lateral (x–y) distance of each cell centroid to the convex hull of all segmented cells. (The top surface, L1, is the real tissue surface; the lower edge at L4 is also artificial but is not modelled here.)

In [ ]:
def lateral_distance(xy):
    hull = Polygon(xy[ConvexHull(xy).vertices])
    return np.array([hull.exterior.distance(Point(p)) for p in xy])


def cell_table(G, pos, topo=None):
    nodes = list(G.nodes)
    df = pd.DataFrame(index=pd.Index(nodes, name="label"))
    df[["x", "y", "z", "area", "volume", "anisotropy", "layer_from_start", "layer_from_end"]] = pos.loc[
        nodes, ["x", "y", "z", "area", "volume", "anisotropy", "layer_from_start", "layer_from_end"]].values
    df["d"] = lateral_distance(df[["x", "y"]].values)
    bc = nx.betweenness_centrality(G)
    df["log_bc"] = np.log10(pd.Series(bc).loc[nodes].values + 1e-9)
    df["degree"] = pd.Series(dict(G.degree())).loc[nodes].values
    # the paper's "random walk centrality" is Newman's random-walk (current-flow) betweenness
    giant = G.subgraph(max(nx.connected_components(G), key=len))  # current-flow betweenness needs a connected graph
    df["rwc"] = pd.Series(nx.current_flow_betweenness_centrality(giant)).reindex(nodes).values
    df["central"] = False
    if topo is not None:
        common = df.index.intersection(topo.index)
        df.loc[common, "central"] = True
        for col in ["Division", "Volume", "Layer", "Log_Betweenness_Centrality", "Random_Walk_Centrality", "Area", "Anisotropy"]:
            df.loc[common, col] = topo.loc[common, col]
    return df


tables = {key: cell_table(s["G"], s["pos"], s["topo"]) for key, s in samples.items()}
check = pd.concat(tables.values()).dropna(subset=["Log_Betweenness_Centrality"])
# which end of the stack is the meristem surface: the layer count that reproduces the published layers
match = {c: (check[c] == check.Layer).mean() for c in ["layer_from_start", "layer_from_end"]}
LAYER = max(match, key=match.get)
# the column count is only approximate (the dome's flanks cross more cells per column), so the
# published layers are used wherever they exist, i.e. for the central cells
for t in tables.values():
    t["layer"] = t["Layer"].fillna(t[LAYER]) if "Layer" in t else t[LAYER]
check["layer"] = check["Layer"]
print("layer agreement with the paper:", {k: round(v, 3) for k, v in match.items()},
      "-> using", LAYER, "for non-central cells, published layers for central cells")
pd.DataFrame({name: [round(spearmanr(check[theirs], check[mine])[0], 4)]
              for name, theirs, mine in [("log BC", "Log_Betweenness_Centrality", "log_bc"),
                                         ("random-walk centrality", "Random_Walk_Centrality", "rwc"),
                                         ("area", "Area", "area"), ("volume", "Volume", "volume"),
                                         ("anisotropy", "Anisotropy", "anisotropy")]},
             index=["Spearman with published values"])

### One meristem

Cells coloured by betweenness on the full network; the central cells the paper evaluated are outlined. Betweenness falls towards the image edge, and still rises across the evaluated region.

In [ ]:
key = ("Rep 2", "T1")
df = tables[key]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), gridspec_kw={"width_ratios": [1, 1.15]})
ax = axes[0]
sc = ax.scatter(df.x, df.y, c=df.log_bc, s=12, cmap="viridis", vmin=df.log_bc.quantile(.02), vmax=df.log_bc.quantile(.98))
c = df[df.central]
ax.scatter(c.x, c.y, s=30, facecolor="none", edgecolor="k", lw=0.6, label="evaluated in the paper")
ax.set_aspect("equal"); ax.set_xlabel("x (µm)"); ax.set_ylabel("y (µm)"); ax.legend(loc="lower left", fontsize=8, frameon=False)
plt.colorbar(sc, ax=ax, label="log10 betweenness", shrink=0.8)
ax.set_title(f"{GENOTYPE} {key[0]} {key[1]}: {len(df)} cells")

ax = axes[1]
for k, t in tables.items():
    bins = pd.cut(t.d, np.arange(0, 56, 4))
    m = t.groupby(bins, observed=True).log_bc.median()
    ax.plot([b.mid for b in m.index], m.values, color="#999", lw=0.8)
cent = pd.concat(tables.values())
cent = cent[cent.central]
ax.axvspan(cent.d.quantile(0.02), cent.d.quantile(0.98), color="#2a78b5", alpha=0.12, label="evaluated region (2nd–98th pct)")
ax.set_xlabel("lateral distance to image edge (µm)"); ax.set_ylabel("median log10 betweenness")
ax.set_title("all 12 samples"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout()

## BOSPERRUS correction

The meristem has three boundaries: the lateral edge of the field of view (artificial), the top surface L1 (the real tissue surface) and the bottom below L4 (artificial; the paper uses L4 as a depth buffer and evaluates only L1–L3). Only the lateral edge is corrected here, and **per layer**: each layer gets its own fit of the score against the lateral distance, so the real surface is never treated as a border and the differences between layers stay untouched. The depth boundary is tested separately below.

Fits use the BOSPERRUS models (constant, piecewise linear, exponential saturation, Michaelis–Menten; AIC selection); each cell is shifted to its layer's fitted interior plateau. For log10 betweenness the additive correction is a multiplicative correction of betweenness.

In [ ]:
MIN_LAYER_CELLS = 30


def layer_group(layer):
    return np.minimum(np.asarray(layer, dtype=float), 4)  # L4 and deeper column counts pooled


def correct_by_layer(df, cols, d_col="d"):
    out, fits = df.copy(), {}
    out[cols] = out[cols].astype(float)  # degree is int; corrected values are not
    for layer, part in df.groupby(layer_group(df["layer"])):
        if len(part) < MIN_LAYER_CELLS:
            continue
        for col in cols:
            flow = Flow.from_distances_and_scores(distances=part[d_col].reset_index(drop=True),
                                                  scores=part[[col]].reset_index(drop=True))
            flow.flow()
            out.loc[part.index, col] = flow.observations[f"BOSPERRUS corrected {col}"].values
            fits[(int(layer), col)] = flow.best_fits[col]
    return out, fits


fits, best_fits = [], {}
for key, df in tables.items():
    corr, layer_fits = correct_by_layer(df, ["log_bc"])
    df["log_bc_corr"] = corr["log_bc"]
    best_fits[key] = layer_fits
    for (layer, _), best in layer_fits.items():
        fits.append(dict(rep=key[0], time=key[1], layer=layer, fit=best.name,
                         effect=best.observed_effect_strength, half_life=best.observed_half_life))
fits = pd.DataFrame(fits)
fits.groupby("layer").agg(samples=("fit", "size"), most_common_fit=("fit", lambda s: s.mode()[0]),
                          effect=("effect", "median"), half_life=("half_life", "median")).round(3)

In [ ]:
LAYER_COLORS = {1: "#d1752a", 2: "#3b9c4a", 3: "#8e5bb5"}
fig, axes = plt.subplots(3, 4, figsize=(13, 8), sharex=True, sharey=True)
for ax, (key, df) in zip(axes.flat, tables.items()):
    ax.scatter(df.d, df.log_bc, s=3, color="#bbb", rasterized=True)
    ax.scatter(df.d, df.log_bc_corr, s=3, color="#2a78b5", alpha=0.4, rasterized=True)
    dd = np.linspace(0, df.d.max(), 100)
    for (layer, _), fit in best_fits[key].items():
        if layer in LAYER_COLORS:
            ax.plot(dd, fit.predict(dd), color=LAYER_COLORS[layer], lw=1.3, label=f"fit L{layer}")
    if df.central.any():
        ax.axvspan(df[df.central].d.min(), df[df.central].d.max(), color="#2a78b5", alpha=0.08)
    ax.set_title(f"{key[0]} {key[1]}", fontsize=9)
for ax in axes[-1]: ax.set_xlabel("distance to edge (µm)")
for ax in axes[:, 0]: ax.set_ylabel("log10 BC")
axes[0, 0].legend(frameon=False, fontsize=7)
plt.tight_layout()

### L1 of every meristem, raw vs. corrected betweenness

Top view of the L1 cells (centroids) of all 12 samples, coloured by log10 betweenness on the full network (raw) and after the per-layer BOSPERRUS correction above. One colour scale for all panels (2nd–98th percentile of raw L1 values).

In [ ]:
TEXTWIDTH = (8.5 * 2.54 - 4.0) / 2.54  # manuscript text width (letterpaper, 2 cm margins), inches
l1 = {k: t[layer_group(t.layer) == 1] for k, t in tables.items()}
pooled = pd.concat(l1.values())
vmin, vmax = pooled.log_bc.quantile([0.02, 0.98])
fig, axes = plt.subplots(4, 6, figsize=(TEXTWIDTH, TEXTWIDTH * 0.78), constrained_layout=True)
for n, (key, t) in enumerate(l1.items()):
    for j, (col, lab) in enumerate([("log_bc", "raw"), ("log_bc_corr", "BOSPERRUS")]):
        ax = axes[n // 3, 2 * (n % 3) + j]
        sc = ax.scatter(t.x, t.y, c=t[col], s=7, lw=0, cmap="viridis", vmin=vmin, vmax=vmax, rasterized=True)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        ax.spines[:].set_visible(False)
        ax.set_title(f"{key[0]} {key[1]}, {lab}", fontsize=6)
cb = fig.colorbar(sc, ax=axes, shrink=0.6, pad=0.01)
cb.set_label("log10 betweenness", fontsize=7); cb.ax.tick_params(labelsize=6.5)

## A data-driven buffer instead of a manual one

The paper chose the central region by hand. BOSPERRUS gives a data-driven alternative: fit a piecewise linear model of a score against the distance to the image edge; the knot is where the score reaches its plateau, and only cells beyond it are kept.

The knot depends on the score. Degree is a local measure, so its knot should be short; betweenness depends on whole paths and should need a wider buffer. Both are fitted per meristem on all cells in L1–L3 and compared with the paper's region (its inner edge = the 5th percentile of the central cells' distance to the edge).

In [ ]:
from bosperrus.fit import PiecewiseLinearFit


def pl_knot(df, col):
    fit = PiecewiseLinearFit(df[col].reset_index(drop=True), df["d"].reset_index(drop=True))
    fit.fit()
    return fit.params["piecewise_linear_b"]


rows, buffers = [], {}
for key, t in tables.items():
    if not t.central.any():
        continue
    cells = t[t.layer.between(1, 3)]
    central = set(cells.index[cells.central])
    for col, measure in [("degree", "degree"), ("log_bc", "log10 BC")]:
        knot = pl_knot(cells, col)
        inside = set(cells.index[cells.d > knot])
        buffers[(key, measure)] = inside
        rows.append(dict(rep=key[0], time=key[1], measure=measure, knot_um=round(knot, 1),
                         paper_edge_um=round(cells.loc[list(central), "d"].quantile(0.05), 1),
                         kept=f"{len(inside) / len(cells):.0%}", paper_kept=f"{len(central) / len(cells):.0%}",
                         jaccard=round(len(inside & central) / len(inside | central), 2),
                         paper_cells_inside=f"{len(inside & central) / len(central):.0%}",
                         inside_with_labels=f"{len(inside & central) / max(len(inside), 1):.0%}"))
knots = pd.DataFrame(rows)
knots

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
order = [f"{r} {t}" for r, t in knots[["rep", "time"]].drop_duplicates().values]
x = np.arange(len(order))
for measure, color, dx in [("degree", "#d1752a", -0.15), ("log10 BC", "#2a78b5", 0.15)]:
    k = knots[knots.measure == measure]
    ax.scatter(x + dx, k.knot_um, color=color, s=30, label=f"PL knot, {measure}", zorder=3)
paper = knots.drop_duplicates(["rep", "time"]).paper_edge_um
ax.scatter(x, paper, marker="_", s=400, color="k", lw=2, label="paper: inner edge of central region")
ax.set_xticks(x, order, rotation=45, ha="right"); ax.set_ylabel("distance to image edge (µm)")
ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()

Division labels exist only for the paper's central cells, so within each data-driven buffer the betweenness–division association can only be tested on the cells that are also central (`inside_with_labels` above says how many that are).

In [ ]:
labelled = pd.concat([t.assign(rep=k[0], time=k[1]) for k, t in tables.items()])
labelled = labelled[labelled.central & labelled.layer.between(1, 3)].copy()
labelled["Division"] = labelled.Division.astype(int)
rows = []
for name, keep in [("paper (central cells)", None), ("PL knot, degree", "degree"), ("PL knot, log10 BC", "log10 BC")]:
    sub = labelled if keep is None else labelled[[lab in buffers[((r, t), keep)] for lab, r, t in zip(labelled.index, labelled.rep, labelled.time)]]
    res = smf.logit("Division ~ log_bc + np.log(Volume) + C(rep) + C(time)", sub).fit(disp=0)
    rows.append(dict(buffer=name, cells=len(sub), divisions=int(sub.Division.sum()),
                     coef=round(res.params["log_bc"], 3), p=f"{res.pvalues['log_bc']:.2g}"))
pd.DataFrame(rows)

## Where do dividing cells sit?

If dividing cells sit at a different distance from the image edge than non-dividing ones, the raw betweenness–division association is partly positional.

In [ ]:
cent = pd.concat([t.assign(rep=k[0], time=k[1]) for k, t in tables.items()])
cent = cent[cent.central].copy()
cent["Division"] = cent.Division.astype(int)
div, non = cent[cent.Division == 1].d, cent[cent.Division == 0].d
print(f"distance to edge: dividing median {div.median():.1f} µm (n={len(div)}), non-dividing {non.median():.1f} µm (n={len(non)}), "
      f"Mann-Whitney p = {mannwhitneyu(div, non).pvalue:.2g}")
fig, ax = plt.subplots(figsize=(4, 3))
ax.boxplot([non, div], widths=0.5, showfliers=False)
ax.set_xticks([1, 2], ["not dividing", "dividing"]); ax.set_ylabel("distance to image edge (µm)")
plt.tight_layout()

## The betweenness–division association, raw vs. corrected

Logistic regression of division on log betweenness over the central cells, with replicate and time point as fixed effects; with and without log cell volume (the paper attributes the association largely to cell size).

In [ ]:
rows = []
for score in ["log_bc", "log_bc_corr"]:
    for extra in ["", " + np.log(Volume)", " + np.log(Volume) + C(Layer)"]:
        r = smf.logit(f"Division ~ {score}{extra} + C(rep) + C(time)", cent).fit(disp=0)
        lo, hi = r.conf_int().loc[score]
        rows.append(dict(score="raw" if score == "log_bc" else "BOSPERRUS", covariates=extra.strip(" +") or "-",
                         coef=round(r.params[score], 3), ci=f"[{lo:.2f}, {hi:.2f}]", p=f"{r.pvalues[score]:.2g}"))
assoc = pd.DataFrame(rows)
assoc

## Crop experiment: a smaller field of view

The full segmented network serves as the reference. Each network is cropped by removing all cells within `delta` µm of its edge, as if the field of view had been smaller. Betweenness is recomputed on the cropped network, raw and BOSPERRUS-corrected against the distance to the *new* edge, and compared with the reference values of the cells the paper evaluated.

In [ ]:
def crop_scores(G, df, delta):
    keep = df.index[df.d > delta]
    H = G.subgraph(keep).copy()
    H = H.subgraph(max(nx.connected_components(H), key=len)).copy()
    sub = df.loc[list(H.nodes)].copy()
    sub["d_crop"] = lateral_distance(sub[["x", "y"]].values)
    sub["crop_raw"] = np.log10(pd.Series(nx.betweenness_centrality(H)).loc[sub.index].values + 1e-9)
    sub["crop_corr"] = correct_by_layer(sub, ["crop_raw"], d_col="d_crop")[0]["crop_raw"]
    return sub


crops, cropped = [], {}
for key, df in tables.items():
    if not df.central.any():
        continue
    for delta in [5, 10, 15]:
        sub = crop_scores(samples[key]["G"], df, delta)
        c = sub[sub.central]
        crops.append(dict(rep=key[0], time=key[1], delta=delta, central_kept=len(c),
                          rho_raw=spearmanr(c.log_bc, c.crop_raw)[0], rho_corr=spearmanr(c.log_bc, c.crop_corr)[0]))
        cropped[(key, delta)] = sub
crops = pd.DataFrame(crops)
crops.groupby("delta")[["central_kept", "rho_raw", "rho_corr"]].median().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.2))
for i, delta in enumerate([5, 10, 15]):
    c = crops[crops.delta == delta]
    ax.scatter(np.full(len(c), i - 0.12) + np.random.default_rng(0).uniform(-0.04, 0.04, len(c)), c.rho_raw, color="#999", s=14)
    ax.scatter(np.full(len(c), i + 0.12) + np.random.default_rng(1).uniform(-0.04, 0.04, len(c)), c.rho_corr, color="#2a78b5", s=14)
ax.set_xticks(range(3), ["5 µm", "10 µm", "15 µm"]); ax.set_xlabel("field of view reduced by")
ax.set_ylabel("Spearman ρ with full-network BC\n(central cells)"); ax.legend(["raw", "BOSPERRUS"], frameon=False, fontsize=8)
plt.tight_layout()

In [ ]:
rows = []
for delta in [5, 10, 15]:
    sub = pd.concat([s.assign(rep=k[0], time=k[1]) for (k, d), s in cropped.items() if d == delta])
    sub = sub[sub.central].copy()
    sub["Division"] = sub.Division.astype(int)
    for score, label in [("log_bc", "full network (reference)"), ("crop_raw", "cropped, raw"), ("crop_corr", "cropped, BOSPERRUS")]:
        r = smf.logit(f"Division ~ {score} + np.log(Volume) + C(rep) + C(time)", sub).fit(disp=0)
        rows.append(dict(delta=delta, score=label, n=len(sub), coef=round(r.params[score], 3), p=f"{r.pvalues[score]:.2g}"))
pd.DataFrame(rows)

## Depth: does the bottom edge matter?

The bottom of the segmented region (below L4) is artificial too, but it coincides with the layer structure, so it cannot be corrected against a distance without also removing real differences between layers. Instead, its size is tested the same way as the lateral crop: L4 is removed from the network, degree, betweenness and RWC are recomputed on L1–L3 alone, and compared with the full-network values. If L3 changes much more than L1 and L2, the depth edge matters; if not, the paper's L4 buffer is enough.

Non-central cells carry the approximate column-count layer, so the removed "L4" is approximate outside the central region. Betweenness and RWC are normalised by network size, so for them the ranks (Spearman) are the fair comparison.

In [ ]:
depth_rows, depth_scores = [], {}
for key, df in tables.items():
    if not df.central.any():
        continue
    G = samples[key]["G"]
    H = G.subgraph(df.index[layer_group(df.layer) <= 3])
    H = H.subgraph(max(nx.connected_components(H), key=len)).copy()
    nodes = list(H.nodes)
    sub = df.loc[nodes].copy()
    sub["degree_noL4"] = pd.Series(dict(H.degree())).loc[nodes].values
    sub["log_bc_noL4"] = np.log10(pd.Series(nx.betweenness_centrality(H)).loc[nodes].values + 1e-9)
    sub["rwc_noL4"] = pd.Series(nx.current_flow_betweenness_centrality(H)).loc[nodes].values
    depth_scores[key] = sub
    for layer in [1, 2, 3]:
        for subset in ["all cells", "central cells"]:
            m = (sub.layer == layer) & (sub.central if subset == "central cells" else True)
            if m.sum() < 10:
                continue
            s = sub[m]
            depth_rows.append(dict(rep=key[0], time=key[1], layer=layer, subset=subset, n=int(m.sum()),
                                   degree_lost=(s.degree - s.degree_noL4).mean(),
                                   rho_bc=spearmanr(s.log_bc, s.log_bc_noL4)[0],
                                   rho_rwc=spearmanr(s.rwc, s.rwc_noL4)[0]))
depth_tab = pd.DataFrame(depth_rows)
depth_tab.groupby(["subset", "layer"])[["n", "degree_lost", "rho_bc", "rho_rwc"]].median().round(3)

In [ ]:
sub = pd.concat([s.assign(rep=k[0], time=k[1]) for k, s in depth_scores.items()])
sub = sub[sub.central].copy()
sub["Division"] = sub.Division.astype(int)
rows = []
for score, label in [("log_bc", "full network (with L4)"), ("log_bc_noL4", "without L4")]:
    r = smf.logit(f"Division ~ {score} + np.log(Volume) + C(rep) + C(time)", sub).fit(disp=0)
    rows.append(dict(network=label, n=len(sub), coef=round(r.params[score], 3), p=f"{r.pvalues[score]:.2g}"))
pd.DataFrame(rows)

## Their Figure 3D under three scenarios

Fig. 3D of the paper shows pairwise Pearson R² between cell geometry and topology, for the central cells of the four T0 meristems (n = 614; layers L1–L3). Here the same matrix is computed for

1. **all cells, raw**: every segmented cell in L1–L3, no buffer, no correction;
2. **central cells, raw**: what the paper shows;
3. **all cells, BOSPERRUS**: every cell in L1–L3, with the network properties (degree, log10 BC, RWC) corrected per meristem and layer against the lateral distance to the image edge. Area, volume and anisotropy stay raw: they describe the cell itself, not its position in a cropped graph.

Geometry and random-walk centrality for the non-central cells are computed here (segmentation and network), and match the published values for the central cells (table above).

In [ ]:
PROPS = {"area": "area", "volume": "volume", "anisotropy": "anisotropy", "degree": "degree",
         "log_bc": "log10 BC", "rwc": "RWC"}


STRUCTURAL = ["degree", "log_bc", "rwc"]  # network properties; cell geometry is not a graph-construction artefact


def corrected(df):
    return correct_by_layer(df, STRUCTURAL)[0]


t0 = {k: t[t.layer.between(1, 3)] for k, t in tables.items() if k[1] == "T0"}
scenarios = {
    "all cells, raw": pd.concat(t0.values()),
    "central cells, raw (paper)": pd.concat([t[t.central] for t in t0.values()]),
    "all cells, BOSPERRUS": pd.concat([corrected(t) for t in t0.values()]),
}
r2 = {name: df[list(PROPS)].corr(method="pearson") ** 2 for name, df in scenarios.items()}
{name: len(df) for name, df in scenarios.items()}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
labels = list(PROPS.values())
for ax, (name, m) in zip(axes, r2.items()):
    mask = np.triu(np.ones_like(m, dtype=bool), k=1)
    im = ax.imshow(np.ma.masked_array(m.values, mask), cmap="magma_r", vmin=0, vmax=1)
    for i in range(len(m)):
        for j in range(i + 1):
            ax.text(j, i, f"{m.values[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if m.values[i, j] > 0.55 else "black")
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right"); ax.set_yticks(range(len(labels)), labels)
    ax.set_title(f"{name}\n(n = {len(scenarios[name])})", fontsize=10)
    ax.spines[:].set_visible(False)
fig.colorbar(im, ax=axes, shrink=0.8, label="Pearson R²")

In [ ]:
# how far each scenario is from the paper's central-cell matrix (lower triangle, off-diagonal)
ref = r2["central cells, raw (paper)"]
tri = np.tril_indices(len(ref), k=-1)
pd.DataFrame({name: {"mean |ΔR²| vs. paper": np.abs(m.values[tri] - ref.values[tri]).mean().round(3),
                     "max |ΔR²| vs. paper": np.abs(m.values[tri] - ref.values[tri]).max().round(3)}
              for name, m in r2.items() if name != "central cells, raw (paper)"})

### Degree vs. cell size: no buffer, the paper's buffer, the BOSPERRUS buffer

Same T0 meristems and layers (L1–L3) as above. The BOSPERRUS buffer keeps the cells beyond the piecewise-linear knot of degree vs. distance to the image edge, fitted per meristem (table in the buffer section).

In [ ]:
TEXTWIDTH = (8.5 * 2.54 - 4.0) / 2.54  # manuscript text width (letterpaper, 2 cm margins), inches
SIZE_PROPS = {"area": "area", "volume": "volume", "degree": "degree"}
size_scenarios = {
    "all cells, raw": pd.concat(t0.values()),
    "central cells, raw (paper)": pd.concat([t[t.central] for t in t0.values()]),
    "BOSPERRUS buffer (degree knot)": pd.concat([t[t.index.isin(buffers[(k, "degree")])] for k, t in t0.items()]),
}
size_r2 = {name: df[list(SIZE_PROPS)].astype(float).corr(method="pearson") ** 2 for name, df in size_scenarios.items()}

fig, axes = plt.subplots(1, 3, figsize=(TEXTWIDTH, TEXTWIDTH * 0.36), constrained_layout=True)
labels = list(SIZE_PROPS.values())
for ax, (name, m) in zip(axes, size_r2.items()):
    mask = np.triu(np.ones_like(m, dtype=bool), k=1)
    im = ax.imshow(np.ma.masked_array(m.values, mask), cmap="magma_r", vmin=0, vmax=1)
    for i in range(len(m)):
        for j in range(i + 1):
            ax.text(j, i, f"{m.values[i, j]:.2g}", ha="center", va="center", fontsize=7,
                    color="white" if m.values[i, j] > 0.55 else "black")
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(labels)), labels, fontsize=7)
    ax.set_title(f"{name}\n(n = {len(size_scenarios[name])})", fontsize=7.5)
    ax.spines[:].set_visible(False)
cb = fig.colorbar(im, ax=axes, shrink=0.8)
cb.set_label("Pearson R²", fontsize=7); cb.ax.tick_params(labelsize=6.5)
fig.savefig("../../result_plots/physically_realized_graphs/meristem_degree_size_r2.svg", dpi=300)
pd.DataFrame({name: {"degree–area": m.loc["degree", "area"], "degree–volume": m.loc["degree", "volume"], "cells": len(size_scenarios[name])}
              for name, m in size_r2.items()}).round(2)

### Betweenness vs. cell size: no buffer, the paper's buffer, all cells corrected, BOSPERRUS buffer

Same T0 meristems and layers (L1–L3). Log10 betweenness in the third scenario is BOSPERRUS-corrected per meristem and layer against the lateral distance to the image edge (as in the Fig. 3D section); area and volume stay raw. The fourth scenario keeps, uncorrected, the cells beyond the per-meristem piecewise-linear knot of log10 betweenness vs. distance to the image edge.

In [ ]:
import textwrap

BC_PROPS = {"area": "area", "volume": "volume", "log_bc": "log10 BC"}
bc_scenarios = {name: scenarios[name] for name in ["all cells, raw", "central cells, raw (paper)", "all cells, BOSPERRUS"]}
bc_scenarios["BOSPERRUS buffer (log10 BC knot)"] = pd.concat([t[t.index.isin(buffers[(k, "log10 BC")])] for k, t in t0.items()])
bc_r2 = {name: df[list(BC_PROPS)].astype(float).corr(method="pearson") ** 2 for name, df in bc_scenarios.items()}

fig, axes = plt.subplots(1, 4, figsize=(TEXTWIDTH, TEXTWIDTH * 0.3), constrained_layout=True)
labels = list(BC_PROPS.values())
for ax, (name, m) in zip(axes, bc_r2.items()):
    mask = np.triu(np.ones_like(m, dtype=bool), k=1)
    im = ax.imshow(np.ma.masked_array(m.values, mask), cmap="magma_r", vmin=0, vmax=1)
    for i in range(len(m)):
        for j in range(i + 1):
            ax.text(j, i, f"{m.values[i, j]:.2g}", ha="center", va="center", fontsize=7,
                    color="white" if m.values[i, j] > 0.55 else "black")
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(labels)), labels, fontsize=7)
    ax.set_title(textwrap.fill(name, 22) + f"\nn = {len(bc_scenarios[name])}", fontsize=7)
    ax.spines[:].set_visible(False)
cb = fig.colorbar(im, ax=axes, shrink=0.8)
cb.set_label("Pearson R²", fontsize=7); cb.ax.tick_params(labelsize=6.5)
fig.savefig("../../result_plots/physically_realized_graphs/meristem_bc_size_r2.svg", dpi=300)
pd.DataFrame({name: {"log10 BC–area": m.loc["log_bc", "area"], "log10 BC–volume": m.loc["log_bc", "volume"], "cells": len(bc_scenarios[name])}
              for name, m in bc_r2.items()}).round(2)

### One meristem, L1: degree and betweenness with the BOSPERRUS buffer

Rep 2 T1, L1 cells (centroids, top view); degree and log10 betweenness are computed on the full network. Blue (the piecewise-linear colour of the fit palette): cells within the BOSPERRUS buffer, i.e. closer to the image edge than the piecewise-linear knot of that score vs. distance, fitted on L1–L3 of this meristem; excluded. Black line: convex hull of the central L1 cells the paper evaluated. Right column: the piecewise-linear fits the knots come from (L1–L3, bands: 5–95 % and 25–75 % per distance bin, black: bin mean).

In [ ]:
import json
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from bosperrus.plotting import plot_fit

PL_BLUE = json.load(open(ROOT / "fit_palette.json"))["Piecewise Linear Fit"]
key = ("Rep 2", "T1")
full = tables[key]
fit_cells = full[full.layer.between(1, 3)]  # the cells the knots are fitted on (buffer section)
t = full[layer_group(full.layer) == 1]
cen = t[t.central][["x", "y"]].values
hull = cen[ConvexHull(cen).vertices]
hull = np.vstack([hull, hull[:1]])

fig, axes = plt.subplots(2, 3, figsize=(TEXTWIDTH, TEXTWIDTH * 0.62), constrained_layout=True,
                         gridspec_kw={"width_ratios": [1, 1, 1.1]})
s = 16
for row, (col, measure, label) in enumerate([("degree", "degree", "degree"), ("log_bc", "log10 BC", "log10 betweenness")]):
    excluded = ~t.index.isin(buffers[(key, measure)])
    kw = dict(cmap="YlOrRd", vmin=t[col].quantile(0.02), vmax=t[col].quantile(0.98), s=s, lw=0, rasterized=True)
    sc = axes[row, 0].scatter(t.x, t.y, c=t[col], **kw)
    axes[row, 0].set_title(f"{'AD'[row]}  {label}", loc="left", fontsize=7.5)
    ax = axes[row, 1]
    ax.scatter(t.x[~excluded], t.y[~excluded], c=t[col][~excluded], **kw)
    ax.scatter(t.x[excluded], t.y[excluded], color=PL_BLUE, s=s, lw=0, rasterized=True)
    ax.plot(hull[:, 0], hull[:, 1], color="k", lw=0.8)
    ax.set_title(f"{'BE'[row]}  buffer ({excluded.sum()} of {len(t)} excluded)", loc="left", fontsize=7.5)
    for a in axes[row, :2]:
        a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([]); a.spines[:].set_visible(False)
    cb = fig.colorbar(sc, ax=axes[row, :2], shrink=0.8, pad=0.02)
    cb.set_label(label, fontsize=7); cb.ax.tick_params(labelsize=6.5)
    # fit row
    fit = PiecewiseLinearFit(fit_cells[col].reset_index(drop=True), fit_cells["d"].reset_index(drop=True))
    fit.fit()
    knot = fit.params["piecewise_linear_b"]
    ax = axes[row, 2]
    plot_fit(ax, fit_cells["d"].values, fit_cells[col].values, fit.predict, bins=25,
             line_kwargs=dict(color=PL_BLUE, linewidth=1.2, label="piecewise linear fit"))
    ax.axvspan(0, knot, color=PL_BLUE, alpha=0.12, lw=0)
    ax.axvline(knot, color=PL_BLUE, lw=0.8, ls="--")
    ax.text(knot, 0.97, f" knot {knot:.2g} µm", transform=ax.get_xaxis_transform(), va="top", fontsize=6, color=PL_BLUE)
    ax.set_xlabel("distance to image edge (µm)", fontsize=7); ax.set_ylabel(label, fontsize=7)
    ax.tick_params(labelsize=6.5)
    ax.set_title(f"{'CF'[row]}  {measure} vs. distance", loc="left", fontsize=7.5)
axes[0, 0].plot([t.x.min(), t.x.min() + 20], [t.y.min() - 4] * 2, color="k", lw=1.2)  # 20 µm scale bar
axes[0, 0].text(t.x.min() + 10, t.y.min() - 6, "20 µm", ha="center", va="top", fontsize=6)
fig.legend(handles=[Patch(color=PL_BLUE, label="BOSPERRUS buffer (closer than the knot), excluded"),
                    Line2D([], [], color="k", lw=0.8, label="evaluated in the paper")],
           loc="outside lower center", ncol=2, frameon=False, fontsize=6, handlelength=1.2)
fig.savefig("../../result_plots/physically_realized_graphs/meristem_L1_maps.svg", dpi=300)

## Their Figure 3A, raw vs. corrected

Fig. 3A compares cell properties of non-dividing cells (T0, n = 582), dividing cells (T0, n = 32) and their daughters (T1, n = 64) as fold changes, computed per meristem and averaged over the four replicates (error bars: SE over replicates; asterisks: t-test p ≤ 0.01).

Here the fold change of a group is its mean divided by the mean of all cells in the scenario at that time point, per meristem. Betweenness enters as BC (not log10), since a fold change of a negative log value has no meaning. Daughters are the T1 cells whose T0 parent divided, from the published lineage files (`*_T0-T1_parents.csv`).

Division and lineage are only published for the central cells, so the groups exist only there. Both scenarios therefore use the same central cells; only the network properties (degree, BC, RWC) differ: raw (as in the paper) or BOSPERRUS-corrected (fitted per meristem on all cells against the lateral distance to the image edge, as in the Fig. 3D section). Area, volume and anisotropy are identical in both.

In [ ]:
def load_parents(zip_path):
    out = {}
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            if name.endswith("_T0-T1_parents.csv"):
                lineage = read_csv_from_zip(z, name)
                lineage.columns = ["a", "b"] + list(lineage.columns[2:])
                out[name.split("/")[1]] = lineage
    return out


parents = {}
for rep, lineage in load_parents(ZIP).items():
    t0, t1 = set(tables[(rep, "T0")].index), set(tables[(rep, "T1")].index)
    # orient the two columns by which time point their labels belong to
    a_in_t1 = np.mean([v in t1 for v in lineage.a]) > np.mean([v in t0 for v in lineage.a])
    parents[rep] = pd.DataFrame({"child": lineage.a if a_in_t1 else lineage.b, "parent": lineage.b if a_in_t1 else lineage.a})

corr_tables = {k: corrected(t[t.layer.between(1, 3)]) for k, t in tables.items() if k[1] in ("T0", "T1")}
FC_PROPS = {"area": "area", "volume": "volume", "anisotropy": "anisotropy", "degree": "degree", "bc": "BC", "rwc": "RWC"}


def scenario_cells(key, scenario):
    t = corr_tables[key] if scenario == "central cells, BOSPERRUS" else tables[key][tables[key].layer.between(1, 3)]
    t = t[t.central].copy()
    t["bc"] = 10 ** t["log_bc"]
    return t


groups = {}
for scenario in ["central cells, raw (paper)", "central cells, BOSPERRUS"]:
    per_rep = []
    for rep in sorted(parents):
        t0, t1 = scenario_cells((rep, "T0"), scenario), scenario_cells((rep, "T1"), scenario)
        dividing = set(t0.index[t0.Division == 1])
        daughters = set(parents[rep].child[parents[rep].parent.isin(dividing)]) & set(t1.index)
        for group, cells, ref in [("non-dividing (T0)", t0.drop(index=list(dividing)), t0),
                                  ("dividing (T0)", t0.loc[list(dividing)], t0),
                                  ("divided (T1)", t1.loc[list(daughters)], t1)]:
            fc = cells[list(FC_PROPS)].mean() / ref[list(FC_PROPS)].mean()
            per_rep.append(dict(rep=rep, group=group, n=len(cells), **fc.to_dict(), cells=cells[list(FC_PROPS)]))
    groups[scenario] = per_rep

pd.DataFrame([{**{k: v for k, v in r.items() if k != "cells"}, "scenario": s} for s, rs in groups.items() for r in rs]) \
  .groupby(["scenario", "group"])["n"].sum().unstack()

In [ ]:
from scipy.stats import ttest_ind

GROUP_COLORS = {"non-dividing (T0)": "#bbbbbb", "dividing (T0)": "#d1752a", "divided (T1)": "#2a78b5"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (scenario, rows) in zip(axes, groups.items()):
    df = pd.DataFrame(rows)
    x = np.arange(len(FC_PROPS))
    for i, group in enumerate(GROUP_COLORS):
        g = df[df.group == group]
        mean, se = g[list(FC_PROPS)].mean(), g[list(FC_PROPS)].sem()
        ax.bar(x + (i - 1) * 0.27, mean, 0.27, yerr=se, color=GROUP_COLORS[group], label=group, capsize=2)
        if group != "non-dividing (T0)":
            ref_cells = pd.concat(df[df.group == "non-dividing (T0)"].cells.tolist())
            grp_cells = pd.concat(g.cells.tolist())
            for j, prop in enumerate(FC_PROPS):
                if ttest_ind(grp_cells[prop], ref_cells[prop], equal_var=False).pvalue <= 0.01:
                    ax.text(j + (i - 1) * 0.27, mean[prop] + se[prop] + 0.03, "*", ha="center", fontsize=12)
    ax.axhline(1, color="k", lw=0.6)
    ax.set_xticks(x, list(FC_PROPS.values())); ax.set_title(scenario, fontsize=10)
axes[0].set_ylabel("mean fold change"); axes[-1].legend(frameon=False, fontsize=8)
plt.tight_layout()

Asterisks: Welch t-test of the cells of a group against the non-dividing cells, pooled over replicates, p ≤ 0.01 (the paper does not say which t-test it used).

## Summary

A positive example for the data-driven buffer: excluding the cells closer to the image edge than the piecewise-linear knot reproduces the paper's central-cell results with far more cells. Additive correction is not the right tool for relations between cell properties here.

- **Border effect.** Betweenness at the image edge is 10–100× lower than inside, in all 12 meristems. Best fits are Michaelis–Menten or exponential saturation with half-lives of 5–10 % of the maximum distance (2–4 µm); the curves reach their plateau after about 15–20 µm.
- **Their buffer.** The evaluated central cells (12–45 µm from the edge) lie on the plateau, at the price of discarding ~80 % of the segmented cells.
- **Data-driven buffer.** Piecewise-linear knots are 5–9 µm and keep ~75 % of the cells. On the T0 meristems (L1–L3), the knot buffer gives degree–area / degree–volume R² of 0.54 / 0.50 (paper's central cells 0.56 / 0.54, all cells 0.44 / 0.43) and log10 BC–area / –volume R² of 0.48 / 0.45 (paper 0.49 / 0.46, all cells 0.35 / 0.33), with 2,283 and 2,236 instead of 614 cells.
- **Correction instead of buffer.** Correcting degree, log10 BC and RWC for all cells moves the Fig. 3D correlations further from the paper (log10 BC–area 0.29). The division association (central cells, 94 divisions in 1,259 cells, with log volume) is 0.69 (p = 0.004) raw and 0.56 (p = 0.003) corrected.
- **Smaller field of view.** Cropping by 15 µm lowers the raw coefficient to 0.58 (p = 0.04; full network 0.68); BOSPERRUS gives 0.66 (p = 0.003). For 5 and 10 µm crops the raw coefficient is unchanged (0.72, 0.69) and BOSPERRUS lowers it (0.45, 0.54).
- **Depth.** Removing L4 lowers the division coefficient from 0.69 to 0.54 (p = 0.02), so the bottom edge matters as well.
- Caveat: a few edge cells have betweenness 0 (log value −9 here); they pull the fits down at the very edge but do not affect the evaluated region.